# Zhuangzhou Colab GPU Server (v0.9.2)

**Purpose**: Persistent PLM inference (ESM-2 + ESM-1v) for local Zhuangzhou agent via HTTP tunnel.

**Architecture**:
```
Local Zhuangzhou ---HTTP---> cloudflared tunnel ---> FastAPI (this notebook)
                                                     +-- /esm_scan           (single-model scan)
                                                     +-- /esm_consensus_scan (multi-model consensus)
                                                     +-- /gpu_info           (GPU status)
                                                     +-- /health             (health check)
                                                     +-- /execute            (arbitrary code)
                                                     +-- /unload             (free VRAM)
```

**Available Models** (all 650M parameters, ~2.5GB VRAM each):
- `esm2_t33_650M_UR50D` — General-purpose PLM, trained on UR50D
- `esm1v_t33_650M_UR90S_1..5` — 5 models designed for variant effect prediction (Meier et al. 2021, 41 DMS datasets validated)

**T4 GPU (15GB)** can hold ~4 models simultaneously. When VRAM is tight, oldest models are auto-evicted.

**Instructions**:
1. Ensure **T4 GPU** runtime (Runtime > Change runtime type > T4 GPU)
2. Run cells **in order** (Step 1 → Step 4)
3. Copy the printed `ZZ_TUNNEL_URL=...` to local config
4. Keep this notebook **open**

In [ ]:
# Step 1: Install Dependencies
print("Installing Python packages...")
!pip install -q fair-esm fastapi uvicorn pydantic nest_asyncio

import os
if not os.path.isfile('/usr/local/bin/cloudflared'):
    print("Installing cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
    print("cloudflared installed")
else:
    print("cloudflared already installed")

# Pre-download ESM-2 model (avoids timeout on first API call)
print("\nPre-downloading ESM-2 model...")
try:
    import esm
    esm.pretrained.load_model_and_alphabet('esm2_t33_650M_UR50D')
    print("ESM-2 (650M) ready")
except Exception as e:
    print(f"ESM-2 pre-download note: {e}")

print("\nAll dependencies ready")

In [ ]:
# Step 2: Verify GPU & System Resources
import torch, psutil

print("=== GPU ===")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = props.total_memory / 1e9
    print(f"  Name: {gpu}")
    print(f"  VRAM: {vram:.1f} GB")
    print(f"  CUDA: {torch.version.cuda}")
    max_models = int(vram // 2.5) - 1  # headroom for inference
    print(f"  Max concurrent models: ~{max_models}")
else:
    print("  NO GPU! Go to: Runtime > Change runtime type > T4 GPU")

print("\n=== System RAM ===")
ram = psutil.virtual_memory()
print(f"  Total: {ram.total / 1e9:.1f} GB")
print(f"  Free:  {ram.available / 1e9:.1f} GB")

print("\n=== Available Models ===")
print("  ESM-2:  esm2_t33_650M_UR50D (~2.5 GB)")
print("  ESM-1v: esm1v_t33_650M_UR90S_1..5 (~2.5 GB each, 5 models)")
print("  Mode:   Auto-evict oldest when VRAM full")

## Step 3: Define Server

Defines the FastAPI server with all endpoints.

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | GET | Health check + uptime + GPU status |
| `/gpu_info` | GET | Detailed GPU memory info |
| `/esm_scan` | POST | Single-model mutation scan (ESM-2 or ESM-1v) |
| `/esm_consensus_scan` | POST | Multi-model consensus scan |
| `/execute` | POST | Arbitrary code execution |
| `/models` | GET | List loaded models + available models |
| `/unload` | POST | Free all GPU memory |
| `/preload_esm` | POST | Pre-load a specific ESM model |

In [ ]:
# Step 3: Define GPU Server
import gc, json, logging, os, subprocess, sys, threading, time
from typing import Optional, List
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('zz-colab')
app = FastAPI(title='Zhuangzhou Colab GPU Server', version='0.9.2')

_model_cache = {}
_cache_lock = threading.Lock()
_start_time = time.time()

_VALID_MODELS = {
    'esm2_t33_650M_UR50D',
    'esm1v_t33_650M_UR90S_1', 'esm1v_t33_650M_UR90S_2',
    'esm1v_t33_650M_UR90S_3', 'esm1v_t33_650M_UR90S_4',
    'esm1v_t33_650M_UR90S_5',
}

# ---- VRAM Management ----

def _gpu_mem():
    if not torch.cuda.is_available():
        return {'total': 0, 'used': 0, 'free': 0}
    t = torch.cuda.get_device_properties(0).total_memory
    r = torch.cuda.memory_reserved(0)
    u = torch.cuda.memory_allocated(0)
    return {'total': round(t/1e9,2), 'used': round(u/1e9,2),
            'reserved': round(r/1e9,2), 'free': round((t-r)/1e9,2)}

def _unload_all():
    keys = list(_model_cache.keys())
    for k in keys:
        logger.info(f'Unloading {k}')
        del _model_cache[k]
    if keys:
        gc.collect(); torch.cuda.empty_cache()
        logger.info(f'After unload: {_gpu_mem()}')

# ---- Model Loading ----

def _get_esm(name='esm2_t33_650M_UR50D'):
    key = f'esm:{name}'
    with _cache_lock:
        if key not in _model_cache:
            mem = _gpu_mem()
            n_loaded = len(_model_cache)
            if n_loaded >= 4 or (mem['free'] < 3.5 and n_loaded > 0):
                oldest = next(iter(_model_cache))
                logger.info(f'VRAM tight, unloading {oldest}')
                del _model_cache[oldest]
                gc.collect(); torch.cuda.empty_cache()
            logger.info(f'Loading {name} ...')
            import esm
            model, alphabet = esm.pretrained.load_model_and_alphabet(name)
            dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            model = model.to(dev).eval()
            _model_cache[key] = {'model': model, 'alphabet': alphabet, 'device': dev}
            logger.info(f'{name} loaded. VRAM: {_gpu_mem()}')
    return _model_cache[key]

def _run_esm_scan(model_name, sequence, positions, top_k):
    c = _get_esm(model_name)
    model, alphabet, dev = c['model'], c['alphabet'], c['device']
    _, _, toks = alphabet.get_batch_converter()([('wt', sequence)])
    toks = toks.to(dev)
    with torch.no_grad():
        lp = torch.log_softmax(model(toks)['logits'][0], dim=-1)
    aa = 'ACDEFGHIKLMNPQRSTVWY'
    pos_list = positions or list(range(1, len(sequence)+1))
    muts = {}
    for p in pos_list:
        if p < 1 or p > len(sequence): continue
        wt = sequence[p-1]; wid = alphabet.get_idx(wt)
        for m in aa:
            if m == wt: continue
            s = (lp[p, alphabet.get_idx(m)] - lp[p, wid]).item()
            key = f'{wt}{p}{m}'
            muts[key] = {'position': p, 'wt': wt, 'mut': m,
                         'mutation': key, 'score': round(s, 4)}
    del toks, lp; torch.cuda.empty_cache()
    return muts

# ---- Request schemas ----

class ESMScanReq(BaseModel):
    sequence: str
    positions: Optional[List[int]] = None
    model: str = 'esm2_t33_650M_UR50D'
    top_k: int = 20

class ESMConsensusReq(BaseModel):
    sequence: str
    positions: Optional[List[int]] = None
    models: List[str] = ['esm2_t33_650M_UR50D', 'esm1v_t33_650M_UR90S_1']
    top_k: int = 20

class ExecReq(BaseModel):
    code: str
    timeout: int = 300

# ---- Endpoints ----

@app.get('/health')
def health():
    info = {'status': 'ok', 'version': '0.9.2',
            'uptime_seconds': round(time.time()-_start_time, 1),
            'models_loaded': list(_model_cache.keys()), 'gpu_available': False}
    if torch.cuda.is_available():
        info['gpu_available'] = True
        info['gpu_name'] = torch.cuda.get_device_name(0)
        info.update(_gpu_mem())
        info['cuda_version'] = torch.version.cuda or ''
    return info

@app.get('/gpu_info')
def gpu_info():
    r = {'gpu_available': False}
    if torch.cuda.is_available():
        r['gpu_available'] = True
        r['gpu_name'] = torch.cuda.get_device_name(0)
        r.update(_gpu_mem())
        r['cuda_version'] = torch.version.cuda or ''
        r['models_loaded'] = list(_model_cache.keys())
    return r

@app.post('/esm_scan')
def esm_scan(req: ESMScanReq):
    """Single-model ESM mutation scan."""
    if not req.sequence: raise HTTPException(400, 'No sequence')
    if req.model not in _VALID_MODELS:
        raise HTTPException(400, f'Invalid model: {req.model}')
    t0 = time.time()
    muts = _run_esm_scan(req.model, req.sequence, req.positions, req.top_k)
    mut_list = sorted(muts.values(), key=lambda x: x['score'], reverse=True)
    gn = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    return {'mutations': mut_list[:req.top_k], 'total_mutations_scanned': len(mut_list),
            'n_positions_scanned': len(req.positions or range(1, len(req.sequence)+1)),
            'sequence_length': len(req.sequence),
            'model': req.model, 'device': 'cuda', 'gpu_name': gn,
            'gpu_memory': _gpu_mem(), 'elapsed_seconds': round(time.time()-t0, 2)}

@app.post('/esm_consensus_scan')
def esm_consensus_scan(req: ESMConsensusReq):
    """Multi-model consensus scan. Averages scores across models."""
    if not req.sequence: raise HTTPException(400, 'No sequence')
    for m in req.models:
        if m not in _VALID_MODELS:
            raise HTTPException(400, f'Invalid model: {m}')
    if len(req.models) < 2:
        raise HTTPException(400, 'Need at least 2 models')
    t0 = time.time()
    all_scores = {}
    for model_name in req.models:
        logger.info(f'Consensus: {model_name}...')
        muts = _run_esm_scan(model_name, req.sequence, req.positions, req.top_k)
        for key, info in muts.items():
            if key not in all_scores:
                all_scores[key] = {'position': info['position'], 'wt': info['wt'],
                                   'mut': info['mut'], 'mutation': key, 'scores': []}
            all_scores[key]['scores'].append(info['score'])
    consensus = []
    for key, info in all_scores.items():
        scores = info['scores']
        avg = sum(scores) / len(scores)
        consensus.append({'position': info['position'], 'wt': info['wt'],
                          'mut': info['mut'], 'mutation': key,
                          'consensus_score': round(avg, 4),
                          'per_model_scores': scores, 'n_models': len(scores)})
    consensus.sort(key=lambda x: x['consensus_score'], reverse=True)
    gn = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    return {'mutations': consensus[:req.top_k],
            'total_mutations_scanned': len(consensus),
            'n_positions_scanned': len(req.positions or range(1, len(req.sequence)+1)),
            'sequence_length': len(req.sequence),
            'models': req.models, 'n_models': len(req.models),
            'device': 'cuda', 'gpu_name': gn,
            'gpu_memory': _gpu_mem(), 'elapsed_seconds': round(time.time()-t0, 2)}

@app.post('/execute')
def execute_code(req: ExecReq):
    import io, traceback
    from contextlib import redirect_stdout, redirect_stderr
    so, se = io.StringIO(), io.StringIO()
    try:
        with redirect_stdout(so), redirect_stderr(se):
            exec(req.code, {'__builtins__': __builtins__})
        stdout = so.getvalue(); data = stdout
        lines = stdout.strip().split('\n')
        if lines:
            try: data = json.loads(lines[-1])
            except: pass
        return {'success': True, 'data': data, 'stdout': stdout, 'stderr': se.getvalue()}
    except Exception as e:
        return {'success': False, 'error': str(e), 'traceback': traceback.format_exc(),
                'stdout': so.getvalue(), 'stderr': se.getvalue()}

@app.get('/models')
def list_models():
    return {'models': list(_model_cache.keys()), 'count': len(_model_cache),
            'available': sorted(_VALID_MODELS)}

@app.post('/unload')
def unload():
    _unload_all()
    return {'status': 'ok', 'gpu_memory': _gpu_mem()}

@app.post('/preload_esm')
def preload_esm(model: str = 'esm2_t33_650M_UR50D'):
    if model not in _VALID_MODELS:
        raise HTTPException(400, f'Invalid model: {model}')
    _get_esm(model)
    return {'status': 'loaded', 'model': model, 'gpu_memory': _gpu_mem()}

# Anti-idle keepalive
def _keepalive():
    while True:
        time.sleep(300)
        try:
            if torch.cuda.is_available():
                x = torch.ones(1, device='cuda'); del x
        except: pass
threading.Thread(target=_keepalive, daemon=True).start()

print(f'Server defined: {len(app.routes)} routes')
print('Endpoints: /esm_scan /esm_consensus_scan /health /gpu_info /models /unload')

## Step 4: Start Tunnel & Server

This cell starts the cloudflared tunnel and uvicorn server.

**This cell will keep running** — do not interrupt it.

Copy the printed `ZZ_TUNNEL_URL=...` to your local config (`~/.zhuangzhou/colab_config.json`).

In [ ]:
# Step 4: Start Tunnel & Server (this cell blocks — keep it running!)
import re, socket, subprocess, threading, time

def _find_port(start=8080):
    for p in range(start, start + 20):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.bind(('0.0.0.0', p)); return p
        except OSError: continue
    return 0

PORT = _find_port()
print(f'Port: {PORT}')

# Kill any leftover cloudflared
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

# Start cloudflared tunnel
print('Starting cloudflared tunnel...')
cf = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE)

tunnel_url = None
deadline = time.time() + 60
while time.time() < deadline:
    if cf.poll() is not None:
        err = cf.stderr.read().decode('utf-8', errors='replace')
        print(f'cloudflared exited: {err[:500]}')
        break
    line = cf.stderr.readline().decode('utf-8', errors='replace')
    if not line:
        time.sleep(0.2)
        continue
    m = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
    if m:
        tunnel_url = m.group(1)
        break

threading.Thread(target=lambda: [_ for _ in cf.stderr], daemon=True).start()

if tunnel_url:
    with open('/content/tunnel_url.txt', 'w') as f: f.write(tunnel_url)
    print(f'\n{"="*60}')
    print(f'  ZZ_TUNNEL_URL={tunnel_url}')
    print(f'{"="*60}')
    print(f'\n  Endpoints: /esm_scan /esm_consensus_scan /health /gpu_info /models /unload')
    print(f'  Keep this notebook running!\n')
else:
    print('ERROR: cloudflared tunnel failed')

# Start uvicorn (blocks)
print(f'Starting server on port {PORT}...')
import nest_asyncio; nest_asyncio.apply()
import uvicorn
uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')